In [70]:
import gym
import crafter
import torch.nn as nn
import matplotlib as plt
import torch
import random
from collections import deque
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os

In [71]:
seed = 42

In [72]:
env = crafter.Env(seed=seed)
env = crafter.Recorder(
    env, '../logs',
    save_stats=True,
    save_video=False,
    save_episode=False,
)

In [73]:
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")

In [74]:
obs = env.reset()
env.action_names

['noop',
 'move_left',
 'move_right',
 'move_up',
 'move_down',
 'do',
 'sleep',
 'place_stone',
 'place_table',
 'place_furnace',
 'place_plant',
 'make_wood_pickaxe',
 'make_stone_pickaxe',
 'make_iron_pickaxe',
 'make_wood_sword',
 'make_stone_sword',
 'make_iron_sword']

In [75]:

class CNNEncoder(nn.Module):
    def __init__(self, h, w, c, c1, c2, hidden_size, kernel_size, stride=1, padding=0, dilation=1, groups=1):
        super().__init__()
        self.params = {'h': h, 'w': w, 'c': c, 'c1': c1, 'c2': c2, 'hidden_size': hidden_size}
        self.conv1 = nn.Conv2d(c, c1, kernel_size, stride=stride, padding=padding, dilation=dilation, groups=groups)
        self.conv2 = nn.Conv2d(c1, c2, kernel_size, stride=stride, padding=padding, dilation=dilation, groups=groups)
        self.conv_net = nn.Sequential(self.conv1, nn.ReLU(), self.conv2, nn.ReLU(), nn.AdaptiveAvgPool2d(1))
        self.hidden_linear = nn.Linear(c2, hidden_size)
    
    def forward(self, x):
        conved_x = self.conv_net(x)
        conved_x = torch.squeeze(conved_x)
        y = self.hidden_linear(conved_x)
        return y


In [76]:
model = CNNEncoder(64, 64, 3, 256, 128, 64, 3)
x = torch.randint(0, 255, [3, 64, 64]).to(torch.float32)
y = model(x)
y.shape

torch.Size([64])

In [77]:
class RNNDeepQNet(nn.Module):
    def __init__(self, h, w, c, c1, c2, hidden_size, kernel_size, hs1, hs2, num_actions, stride=1, padding=0, dilation=1, groups=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.cnn_encoder = CNNEncoder(h, w, c, c1, c2, hidden_size, kernel_size, stride, padding, dilation, groups)
        self.hh = nn.Sequential(nn.Linear(hidden_size, hidden_size))
        self.dqn = nn.Sequential(nn.Linear(hidden_size, hs1), nn.ReLU(), nn.Linear(hs1, hs2), nn.ReLU(), nn.Linear(hs2, num_actions), nn.Softmax())
        self.tanh = nn.Tanh()
    
    def forward(self, x):
        dimensions = len(x.shape)
        
        if dimensions == 5:
            hidden_info = torch.zeros(x.shape[0], self.hidden_size).to(device)
            for i in range(3):
                hidden_info == self.tanh(self.hh(hidden_info) + self.cnn_encoder(x[:, i, :, :, :]))
        else:
            hidden_info = torch.zeros(self.hidden_size).to(device)
            for i in range(3):
                hidden_info = self.tanh(self.hh(hidden_info) + self.cnn_encoder(x[i]))

        return self.dqn(hidden_info)


In [78]:
model = RNNDeepQNet(64, 64, 3, 256, 128, 64, 3, 128, 256, 17).to(device)
x = torch.randint(0, 255, [3, 3, 64, 64]).to(torch.float32).to(device)
y = model(x)
y.shape

/opt/homebrew/anaconda3/envs/conda_env/lib/python3.12/site-packages/torch/nn/modules/module.py:1775: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


torch.Size([17])

In [79]:
def eps_greedy(model, state, num_actions, eps=0.3):
    sample_from_model = random.choices([0, 1], weights=[eps, 1 - eps])[0]
    return torch.argmax(model(state)).item() if sample_from_model == 1 else random.choice(range(num_actions))

In [80]:
print(eps_greedy(model, x, 17))

14


In [81]:
def play_and_store(model, env, er_buffer_obs, er_buffer_future_obs, er_buffer_action, er_buffer_reward, er_done, num_actions, eps=0.3):
    obs = env.reset()
    done = False
    stack_frames = deque(maxlen=3)
    obs_empty1 = torch.empty((3, 64, 64), dtype=torch.float32).to(device)
    obs_empty2 = torch.empty((3, 64, 64), dtype=torch.float32).to(device)
    stack_frames.append(obs_empty1)
    stack_frames.append(obs_empty2)
    obs_tensor = torch.permute(torch.tensor(obs).to(torch.float32), (2, 0, 1)).to(device)
    stack_frames.append(obs_tensor)
    while done is not True:
        obs_for_model = torch.stack(list(stack_frames)).to(device)
        action = eps_greedy(model, obs_for_model, num_actions, eps)
        obs, reward, done, info = env.step(action)
        stack_frames.append(torch.permute(torch.tensor(obs).to(torch.float32), (2, 0, 1)).to(device))
        future_obs = torch.stack(list(stack_frames)).to(device)
        er_buffer_obs = torch.cat([er_buffer_obs, obs_for_model.unsqueeze(0).to(device)], dim = 0)
        er_buffer_future_obs = torch.cat([er_buffer_future_obs, future_obs.unsqueeze(0).to(device)], dim = 0)
        er_buffer_action = torch.cat([er_buffer_action, torch.tensor([action]).to(device)], dim = 0)
        er_buffer_reward = torch.cat([er_buffer_reward, torch.tensor([reward]).to(device)], dim = 0)
        er_done = torch.cat([er_done, torch.tensor([1 if done is True else 0]).to(device)])
        
    return er_buffer_obs, er_buffer_future_obs, er_buffer_action, er_buffer_reward, er_done

In [82]:
height = 64
weight = 64
channels_in = 3
channels_hidden_1 = 256
channels_hidden_2 = 128
hidden_size = 128
kernel_size = 3
hidden_size_inside_1 = 128
hidden_size_inside_2 = 256
num_actions = 17
num_games_played = 3000
eps = 1.0
factor_decay_eps = 0.9985
discount_factor = 0.99
steps_updating = 1000

In [83]:
def set_seed(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(True)
    np.random.seed(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = 1
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
    elif torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [84]:
def loss_bellman(target, pred):
    diff = target - pred
    return diff.mean()

In [ ]:
def q_learning(
    env,
    q_network,
    target_q_network,
    num_actions,
    num_games_played,
    eps,
    factor_decay_eps,
    discount_factor,
    steps_updating,
    device
):
    updates = 0
    for _ in range(num_games_played):
        # Init experience replay buffer
        er_buffer_obs = torch.empty(0, 3, 3, 64, 64).to(device)
        er_buffer_future_obs = torch.empty(0, 3, 3, 64, 64).to(device)
        er_buffer_action = torch.empty(0).to(device)
        er_buffer_reward = torch.empty(0).to(device)
        er_done = torch.empty(0).to(device)

        er_buffer_obs, er_buffer_future_obs, er_buffer_action, er_buffer_reward, er_done = play_and_store(q_network, env, er_buffer_obs, er_buffer_future_obs, er_buffer_action, er_buffer_reward, er_done, num_actions, eps)
        len_steps = er_buffer_action.shape[0]
        indices = torch.randperm(len_steps)

        random_er_buffer_obs = er_buffer_obs[indices]
        random_er_buffer_future_obs = er_buffer_future_obs[indices]
        random_er_buffer_action = er_buffer_action[indices]
        random_er_buffer_reward = er_buffer_reward[indices]
        random_er_done = er_done[indices]
        
        optimizer = torch.optim.RMSprop(
            q_network.parameters(),
            lr=2.5e-4,
            centered=True,
            alpha=0.95,
            eps=0.01
        )

        print(len_steps)
        for i in range(0, len_steps, 64):
            optimizer.zero_grad()
            target_q_network.eval()
            pred = torch.max(q_network(random_er_buffer_obs[i:min(i + 64, len_steps)]), dim=1).values
            max_next_state_q = torch.max(target_q_network(random_er_buffer_future_obs[i:min(i + 64, len_steps)]), dim = 1).values
            next_state_q = discount_factor * (torch.tensor(1) - random_er_done[i:min(i + 64, len_steps)]) * max_next_state_q
            target = random_er_buffer_reward[i:min(i + 64, len_steps)] + next_state_q
            loss = loss_bellman(target, pred)
            loss.backward()
            optimizer.step()
            updates += 1
            if updates % steps_updating == 0:
                target_q_network.load_state_dict(q_network.state_dict())
    eps = eps * factor_decay_eps
    return q_network, target_q_network

In [87]:
q_network = RNNDeepQNet(height, weight, channels_in, channels_hidden_1,
                        channels_hidden_2, hidden_size, kernel_size,
                        hidden_size_inside_1, hidden_size_inside_2, num_actions).to(device)

target_q_network = RNNDeepQNet(height, weight, channels_in, channels_hidden_1,
                        channels_hidden_2, hidden_size, kernel_size,
                        hidden_size_inside_1, hidden_size_inside_2, num_actions).to(device)
target_q_network.load_state_dict(q_network.state_dict())

q_learning(env, q_network, target_q_network, num_actions, num_games_played, eps, factor_decay_eps, discount_factor, steps_updating, device)

KeyboardInterrupt: 